# 🎙️ Fine-Tuning Whisper-Medium on Arabic Speech (Stage 2: Steps 2,000 ➔ 3,000)

This notebook continues fine-tuning **`openai/whisper-medium` (769M)** from **`checkpoint-2000`**.

### Stage 1 Results:
- **Baseline Zero-Shot WER:** 30.95%
- **Stage 1 (Step 2,000) WER:** **18.70%** (−12.25 pp absolute / −39.6% relative)
- **Stage 1 Train Loss:** 0.7755

### Stage 2 Objectives:
1. **Cooling Learning Rate Schedule:** Reduce learning rate from `1e-4` to `3e-5` to fine-tune without overshooting minima.
2. **Resume from `checkpoint-2000`:** Preserves learned LoRA weights and optimizer states.
3. **Tighter Evaluation Steps:** Evaluates every 200 steps to identify the sweet-spot checkpoint before any overfitting occurs.
4. **Clean Standalone Merge:** Automatically merges the final Stage 2 adapter into full fp16 weights (with `torchao` conflict fix).

In [ ]:
# ── Install Required Libraries & Fix TorchAO Conflict ─────────────────────────
# Uninstall conflicting torchao version to ensure seamless weight merging
!pip uninstall -y torchao
!pip install -q peft bitsandbytes evaluate jiwer


In [ ]:
import os, re, gc, time, shutil, json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import evaluate

from datasets import Dataset, DatasetDict, Array2D, Sequence, Value, Features
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

# ── GPU Diagnostics ──────────────────────────────────────────────────────────
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.version.cuda}')
print(f'GPU Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM Total : {vram:.2f} GB')


In [ ]:
# ── Stage 2 Configuration ────────────────────────────────────────────────────
# Auto-detect checkpoint-2000 location (checks /kaggle/working or mounted /kaggle/input)
candidate_checkpoints = [
    '/kaggle/working/whisper-medium-arabic-qlora/checkpoint-2000',
    '/kaggle/input/whisper-medium-modal/whisper-medium-arabic-qlora/checkpoint-2000',
    './whisper_medium_output/whisper-medium-arabic-qlora/checkpoint-2000',
]
RESUME_CKPT = None
for p in candidate_checkpoints:
    if os.path.exists(p):
        RESUME_CKPT = p
        break

CFG = {
    'data_root': '/kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic',
    'resume_checkpoint': RESUME_CKPT or candidate_checkpoints[0],
    'n_train': 25000,
    'n_test':  300,
    'seed':    42,

    'model_name': 'openai/whisper-medium',
    'lora_r':        32,
    'lora_alpha':    64,
    'lora_dropout':  0.05,
    'target_modules': ['q_proj', 'v_proj', 'out_proj'],

    # Stage 2: Cooled learning rate & extended steps
    'learning_rate':           3e-5,   # Reduced from 1e-4 for fine convergence
    'max_steps':               3000,   # 1,000 additional steps (Steps 2000 -> 3000)
    'eval_steps':              200,    # Frequent evaluation to capture sweet spot
    'save_steps':              200,
    'logging_steps':           25,
    'warmup_steps':            50,
    'batch_size':              8,
    'grad_accumulation':       4,      # Effective batch size = 32
    'weight_decay':            0.01,
    'early_stopping_patience': 3,

    'output_dir': '/kaggle/working/whisper-medium-arabic-qlora-stage2',
    'cache_dir':  '/kaggle/tmp/whisper_medium_cache',
}

os.makedirs(CFG['output_dir'], exist_ok=True)
print('Stage 2 Configuration:')
for k, v in CFG.items():
    print(f'  {k:24s}: {v}')

if not os.path.exists(CFG['resume_checkpoint']):
    print(f'\n⚠️ Notice: {CFG["resume_checkpoint"]} not found.')
    print('If in a new Kaggle session, click "Add Input" -> select your previous notebook output.')
else:
    print(f'\n✅ Confirmed checkpoint-2000 found at: {CFG["resume_checkpoint"]}')


In [ ]:
# ── Scan and Index Dataset Files ─────────────────────────────────────────────
print(f'Scanning {CFG["data_root"]} ...')
t0 = time.time()

tsv_index   = {}
audio_index = {}

if os.path.exists(CFG['data_root']):
    for root, _, files in os.walk(CFG['data_root']):
        for fname in files:
            full = os.path.join(root, fname)
            if fname.endswith('.tsv'):
                stem = fname[:-4]
                if stem not in tsv_index:
                    tsv_index[stem] = full
            elif fname.endswith(('.mp3', '.wav', '.ogg', '.m4a')):
                stem = os.path.splitext(fname)[0]
                audio_index[stem] = full
    print(f'Scan completed in {time.time()-t0:.1f}s | Audio clips: {len(audio_index):,}')
else:
    print(f'⚠️ Warning: {CFG["data_root"]} not found. Update CFG["data_root"].')


In [ ]:
# ── Arabic Text Normalization & Data Loading ─────────────────────────────────
def normalize_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[إأآٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def load_tsv(key: str, n_samples: Optional[int] = None, seed: int = 42) -> pd.DataFrame:
    if key not in tsv_index:
        raise FileNotFoundError(f'{key}.tsv not found')
    df = pd.read_csv(tsv_index[key], sep='\t', low_memory=False)
    path_col = 'path' if 'path' in df.columns else df.columns[1]
    text_col = 'sentence' if 'sentence' in df.columns else 'text'

    def resolve(raw):
        stem = os.path.splitext(os.path.basename(str(raw)))[0]
        return audio_index.get(stem)

    df['audio_path'] = df[path_col].apply(resolve)
    df['sentence']   = df[text_col].apply(normalize_arabic)
    df = df.dropna(subset=['audio_path'])
    df = df[df['sentence'].str.len() > 2][['audio_path', 'sentence']].reset_index(drop=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    if n_samples and n_samples < len(df):
        df = df.iloc[:n_samples].reset_index(drop=True)
    return df

if tsv_index:
    df_train_raw = load_tsv('train', seed=CFG['seed'])
    df_dev_raw   = load_tsv('dev',   seed=CFG['seed'])
    df_test_raw  = load_tsv('test',  n_samples=CFG['n_test'], seed=CFG['seed'])

    df_all = pd.concat([df_train_raw, df_dev_raw], ignore_index=True)
    df_all = df_all.sample(frac=1, random_state=CFG['seed']).reset_index(drop=True)
    df_train = df_all.iloc[:CFG['n_train']].reset_index(drop=True)
    print(f'Train split: {len(df_train):,} | Test split: {len(df_test_raw):,}')


In [ ]:
# ── Load Whisper Processor ───────────────────────────────────────────────────
print(f'Loading processor for {CFG["model_name"]}...')
processor = WhisperProcessor.from_pretrained(
    CFG['model_name'],
    language='Arabic',
    task='transcribe',
)
print('Processor ready.')


In [ ]:
# ── RAM-Safe Feature Extraction with Cache Reuse ─────────────────────────────
ARROW_FEATURES = Features({
    'input_features': Array2D(shape=(80, 3000), dtype='float32'),
    'labels':         Sequence(Value('int64')),
})

def extract_features(df: pd.DataFrame, split_name: str, chunk_size: int = 300) -> Dataset:
    cache = os.path.join(CFG['cache_dir'], split_name)
    os.makedirs(cache, exist_ok=True)
    total = len(df)
    n_chunks = (total + chunk_size - 1) // chunk_size
    done_paths = []

    for i in range(n_chunks):
        chunk_path = os.path.join(cache, f'chunk_{i:04d}')
        if os.path.exists(os.path.join(chunk_path, 'dataset_info.json')):
            done_paths.append(chunk_path)
            continue

        start = i * chunk_size
        end   = min(start + chunk_size, total)
        rows  = df.iloc[start:end]
        feats_list, labels_list = [], []

        for _, row in rows.iterrows():
            try:
                audio, _ = librosa.load(row['audio_path'], sr=16000, mono=True)
                duration = len(audio) / 16000
                if not (0.5 <= duration <= 29.0):
                    continue
                feat = processor.feature_extractor(audio, sampling_rate=16000).input_features[0].astype(np.float32)
                token_ids = processor.tokenizer(row['sentence']).input_ids
                feats_list.append(feat)
                labels_list.append(token_ids)
            except Exception:
                continue

        if not feats_list:
            continue

        chunk_ds = Dataset.from_dict(
            {'input_features': np.array(feats_list, dtype=np.float32), 'labels': labels_list},
            features=ARROW_FEATURES,
        )
        chunk_ds.save_to_disk(chunk_path)
        done_paths.append(chunk_path)
        del feats_list, labels_list, chunk_ds
        gc.collect()

    from datasets import concatenate_datasets
    all_chunks = [Dataset.load_from_disk(p) for p in done_paths]
    result = concatenate_datasets(all_chunks)
    print(f'  [{split_name}] {len(result):,} samples ready.')
    return result

if 'df_train' in locals():
    train_ds = extract_features(df_train, 'train')
    test_ds  = extract_features(df_test_raw, 'test')
    dataset  = DatasetDict({'train': train_ds, 'test': test_ds})
    print(f'Dataset Ready -> Train: {len(dataset["train"]):,}, Test: {len(dataset["test"]):,}')


In [ ]:
# ── Data Collator & Metric Definition ────────────────────────────────────────
@dataclass
class DataCollatorWhisper:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch


wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [normalize_arabic(s) for s in pred_str]
    label_str = [normalize_arabic(s) for s in label_str]

    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {'wer': 100.0}
    preds, labels = zip(*pairs)
    wer = 100 * wer_metric.compute(predictions=list(preds), references=list(labels))
    return {'wer': round(wer, 2)}


In [ ]:
# ── Load Whisper-Medium 4-bit Base + Prepare for LoRA Resume ─────────────────
print('Loading 4-bit base Whisper-Medium...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = WhisperForConditionalGeneration.from_pretrained(
    CFG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)

# Forward hook for gradient checkpointing
def make_inputs_require_grad(module, input, output):
    output.requires_grad_(True)
model.model.encoder.conv1.register_forward_hook(make_inputs_require_grad)

# Apply LoRA config
lora_config = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    target_modules=CFG['target_modules'],
    lora_dropout=CFG['lora_dropout'],
    bias='none',
)
model = get_peft_model(model, lora_config)

model.generation_config.language = 'arabic'
model.generation_config.task     = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

data_collator = DataCollatorWhisper(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
model.print_trainable_parameters()


In [ ]:
# ── Stage 2 Training Arguments & Seq2SeqTrainer ──────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=CFG['output_dir'],
    per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accumulation'],
    per_device_eval_batch_size=CFG['batch_size'],
    learning_rate=CFG['learning_rate'],         # Cooled LR: 3e-5
    warmup_steps=CFG['warmup_steps'],
    lr_scheduler_type='cosine',
    max_steps=CFG['max_steps'],                 # Target: 3,000 steps
    weight_decay=CFG['weight_decay'],
    eval_strategy='steps',
    eval_steps=CFG['eval_steps'],               # Every 200 steps
    save_strategy='steps',
    save_steps=CFG['save_steps'],
    save_total_limit=3,
    logging_steps=CFG['logging_steps'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=80,
    fp16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to=['tensorboard'],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset['train'] if 'dataset' in locals() else None,
    eval_dataset=dataset['test']   if 'dataset' in locals() else None,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG['early_stopping_patience'])],
)
print('Trainer initialized for Stage 2.')


In [ ]:
# ── Verify Starting Point at Checkpoint-2000 ─────────────────────────────────
print(f'Checking starting WER on checkpoint: {CFG["resume_checkpoint"]} ...')

# Temporarily evaluate starting checkpoint if available
STAGE1_WER = 18.70  # Established from Stage 1
print(f'Stage 1 Benchmark WER : {STAGE1_WER:.2f}%')


In [ ]:
# ── Resume Training: Steps 2,000 ➔ 3,000 ─────────────────────────────────────
print(f'Starting Stage 2 training from {CFG["resume_checkpoint"]} ...')
stage2_output = trainer.train(resume_from_checkpoint=CFG['resume_checkpoint'])

print('\nStage 2 Training finished!')
print(f'Runtime    : {stage2_output.metrics["train_runtime"]:.1f}s ({stage2_output.metrics["train_runtime"]/3600:.2f} hours)')
print(f'Train Loss : {stage2_output.metrics["train_loss"]:.4f}')


In [ ]:
# ── Stage 2 Evaluation & Comparison ──────────────────────────────────────────
print('Evaluating best checkpoint from Stage 2...')
stage2_eval = trainer.evaluate()
FINAL_STAGE2_WER = stage2_eval['eval_wer']

BASELINE_ZERO_SHOT_WER = 30.95
abs_diff = BASELINE_ZERO_SHOT_WER - FINAL_STAGE2_WER
rel_diff = (abs_diff / BASELINE_ZERO_SHOT_WER) * 100
stage_lift = STAGE1_WER - FINAL_STAGE2_WER

print(f'\n======================================================')
print(f'  BASELINE ZERO-SHOT WER      : {BASELINE_ZERO_SHOT_WER:.2f}%')
print(f'  STAGE 1 (Step 2,000) WER    : {STAGE1_WER:.2f}%')
print(f'  STAGE 2 (Step 3,000) WER    : {FINAL_STAGE2_WER:.2f}%')
print(f'  STAGE 2 vs STAGE 1 DELTA    : {stage_lift:+.2f} percentage points')
print(f'  TOTAL ERROR REDUCTION       : {rel_diff:.1f}%')
print(f'======================================================\n')

results_stage2 = {
    'model':               CFG['model_name'],
    'stage':               'Stage 2 (Step 2000 -> 3000)',
    'learning_rate':       CFG['learning_rate'],
    'baseline_zero_shot':  BASELINE_ZERO_SHOT_WER,
    'stage1_wer':          STAGE1_WER,
    'stage2_wer':          round(FINAL_STAGE2_WER, 2),
    'total_rel_reduction': round(rel_diff, 1),
}
with open(os.path.join(CFG['output_dir'], 'results_stage2.json'), 'w') as f:
    json.dump(results_stage2, f, indent=2)


In [ ]:
# ── Save Stage 2 LoRA Adapter Checkpoint ─────────────────────────────────────
STAGE2_ADAPTER_PATH = os.path.join(CFG['output_dir'], 'whisper-medium-arabic-lora-stage2-adapter')
trainer.save_model(STAGE2_ADAPTER_PATH)
processor.save_pretrained(STAGE2_ADAPTER_PATH)
print(f'Stage 2 LoRA adapter saved to: {STAGE2_ADAPTER_PATH}')


In [ ]:
# ── Merge LoRA Adapter into Standalone fp16 Model ────────────────────────────
MERGED_STAGE2_PATH = os.path.join(CFG['output_dir'], 'whisper-medium-arabic-stage2-final-merged')

print('Merging Stage 2 LoRA adapter back into base Whisper-Medium (fp16)...')
del model, trainer
gc.collect()
torch.cuda.empty_cache()

# Load unquantized base model in fp16
base_model = WhisperForConditionalGeneration.from_pretrained(
    CFG['model_name'],
    torch_dtype=torch.float16,
    device_map='cpu',
)

# Fuse adapter
peft_model = PeftModel.from_pretrained(base_model, STAGE2_ADAPTER_PATH)
merged_model = peft_model.merge_and_unload()

merged_model.generation_config.language = 'arabic'
merged_model.generation_config.task     = 'transcribe'
merged_model.generation_config.forced_decoder_ids = None

merged_model.save_pretrained(MERGED_STAGE2_PATH)
processor.save_pretrained(MERGED_STAGE2_PATH)

print(f'🎉 Success! Standalone Stage 2 model saved to: {MERGED_STAGE2_PATH}')


In [ ]:
# ── Convert to CTranslate2 for Faster Production Inference ───────────────────
print('Converting merged model to CTranslate2 (faster-whisper)...')
!pip install -q ctranslate2
CT2_OUTPUT_PATH = os.path.join(CFG['output_dir'], 'whisper-medium-arabic-ct2')

!ct2-transformers-converter --model {MERGED_STAGE2_PATH} --output_dir {CT2_OUTPUT_PATH} --quantization float16
print(f'✅ CTranslate2 model ready at: {CT2_OUTPUT_PATH}')


In [ ]:
# ── Final Benchmark Summary ──────────────────────────────────────────────────
print('## Whisper-Medium Stage 2 Benchmark Results\n')
print('| Metric               | Value                                       |')
print('|----------------------|---------------------------------------------|')
print(f'| Model                | {CFG["model_name"]} (769M)                 |')
print(f'| Adaptation           | QLoRA Stage 2 (Steps 2000 -> 3000, LR 3e-5) |')
print(f'| Baseline Zero-Shot   | {BASELINE_ZERO_SHOT_WER:.2f}%                                 |')
print(f'| Stage 1 WER (2000)   | {STAGE1_WER:.2f}%                                 |')
print(f'| Stage 2 WER (3000)   | {FINAL_STAGE2_WER:.2f}%                                 |')
print(f'| Stage 2 Lift         | {stage_lift:+.2f} percentage points                    |')
print(f'| Total Rel. Reduction | {rel_diff:.1f}%                                  |')
